# M14, 60 km/h ramp: policy seeds 1 and 2 on Google Colab

Adds two policy (training) seeds to the finished seed-0 study in the same Drive folder (`m14_ramp60`).
The initial SUMO dataset, the round-0 DeepONet ensemble and the tuned ALINEA / PI-ALINEA / constant baselines
are shared, so only the seed-dependent training runs:

- **surrogate-PPO** with data aggregation (up to 5 rounds) for seed 1 and seed 2,
- **direct SUMO-PPO** at 1000 episodes for seed 1 and seed 2,

all four at the same time (`run.py seeds`), each PPO process capped at `TORCH_THREADS` CPU threads. Then the final
evaluation covers seeds 0, 1 and 2 (seed 0's results are reused, not rerun) and the tables are rebuilt:
`runs/study/m14/tables/tables.md` becomes the summary over the three seeds (mean ± sd, headline reductions with
confidence intervals over seeds and episodes), and each seed's own tables go to `tables/seed_<s>/`.

**Expected time** (12-CPU L4 runtime): the direct PPO runs are the longest job, about 5-6 h (one simulation at a
time each). The surrogate-PPO loops take about 1-1.5 h each with the faster surrogate environment. The final
evaluation adds about 1 h. Plan for roughly 7 h.

**The code is pinned** to commit `@COMMIT@` (faster surrogate environment, thread cap, `run.py seeds`).

| cell | what | when |
|---|---|---|
| 1 | parameters | every session |
| 2 | Drive + code (pinned commit), keeps a copy of the seed-0 tables | every session |
| 3 | install SUMO + check | every session |
| 4 | launch / resume in the background | first session, and after a lost session |
| 5 | watchdog: keeps the session busy, relaunches a stopped driver | right after cell 4; leave it running |
| 6 | status | any time (stop cell 5 first, then restart it) |
| 7 | tables over the three seeds | at the end |
| 8 | archive of the results on Drive | at the end |

**After a lost session:** rerun cells 1-3, then cell 4 and cell 5. Finished work is reused: the direct PPO runs
continue from their latest checkpoint, an interrupted aggregation round or evaluation is moved aside and redone.

In [ ]:
# 1. Parameters
REPO_URL = "https://github.com/LejunZhou/traffic-surrogate-rl.git"
COMMIT = "c3d07c0b5b780a5611659ca75fb1248ba240a314"                              # pinned code version for the whole seed sweep
WORK = "/content/drive/MyDrive/m14_ramp60"       # the finished seed-0 study (data, ensemble, baselines, seed 0)
NEW_SEEDS = [1, 2]
ROUNDS = 5                                        # aggregation rounds, as for seed 0
BUDGETS = [1000]                                  # direct SUMO-PPO budget in episodes, as for seed 0
TORCH_THREADS = 4                                 # CPU threads per PPO process (4 processes share the machine)

In [ ]:
# 2. Drive + code at the pinned commit
from google.colab import drive
drive.mount("/content/drive")
import os, shutil, subprocess, datetime, json

assert os.path.isdir(WORK), f"{WORK} not found: this notebook extends the finished seed-0 study"
for need in ("data/round0/split_index.json", "runs/deeponet/round0/manifest.json", "runs/study/m14/alinea_tuning.json",
             "runs/aggregation/m14_s0/rounds.json", "runs/study/m14/tables/tables.md"):
    assert os.path.exists(f"{WORK}/{need}"), f"seed-0 study incomplete: {need} missing"

CLONE = "/content/traffic-surrogate-rl"
if not os.path.isdir(CLONE):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, CLONE], check=True)
subprocess.run(["git", "-C", CLONE, "fetch", "--depth", "1", "origin", COMMIT], check=True)
subprocess.run(["git", "-C", CLONE, "checkout", "-q", COMMIT], check=True)

# seed-0-only tables and manifest, kept once before the sweep overwrites them
if not os.path.exists(f"{WORK}/runs/study/m14/tables_seed0_only"):
    shutil.copytree(f"{WORK}/runs/study/m14/tables", f"{WORK}/runs/study/m14/tables_seed0_only")
    shutil.copy2(f"{WORK}/runs/study/m14/arms.json", f"{WORK}/runs/study/m14/arms_seed0_only.json")

version_file = f"{WORK}/CODE_VERSION.txt"
if COMMIT[:7] not in open(version_file).read().splitlines()[-1]:
    for item in ["src", "scripts", "configs", "tests", "docs", "run.py", "pyproject.toml", "README.md"]:
        src, dst = f"{CLONE}/m14/{item}", f"{WORK}/{item}"
        if os.path.isdir(src):
            shutil.copytree(src, dst, dirs_exist_ok=True, ignore=shutil.ignore_patterns("__pycache__", "*.pyc"))
        else:
            shutil.copy2(src, dst)
    with open(version_file, "a") as f:
        f.write(f"{datetime.datetime.now().isoformat(timespec='seconds')}  {COMMIT[:7]}  update (seeds {NEW_SEEDS})\n")
print(open(version_file).read())
os.chdir(WORK)
!grep -c "def predict_step_cached" src/surrogate/deeponet.py
!python run.py seeds --help | head -3

In [ ]:
# 3. Install SUMO and dependencies, check the runtime
!pip install -q "eclipse-sumo==1.27.1" "traci==1.27.1" "sumolib==1.27.1" "stable-baselines3>=2.0" "gymnasium>=0.29" pyyaml
import os, sys, sumo, torch
os.environ["SUMO_HOME"] = os.path.dirname(sumo.__file__)
os.environ["PATH"] = os.path.join(os.environ["SUMO_HOME"], "bin") + os.pathsep + os.environ["PATH"]
os.environ["MPLBACKEND"] = "Agg"
os.chdir(WORK)
!python run.py check
CORES = os.cpu_count()
WORKERS = max(2, CORES // 2)      # SUMO workers per branch; the four branches rarely validate at the same time
print(f"\nCPU cores: {CORES} -> SUMO workers per branch {WORKERS}, torch threads per PPO process {TORCH_THREADS}")

## The seed sweep

Cell 4 starts `run.py seeds` as a background process. Logs: `runs/logs/seeds_driver.log` (driver) and
`runs/logs/pipeline_{surrogate,direct}_s{1,2}.log` (the four training branches). Cell 4 refuses to start a second
driver while one is running.

**Then start cell 5 and leave it running.** Colab disconnects a notebook with no running cell after a while, even if
background processes are busy (seed 0 lost its runtime that way). Cell 5 keeps a cell running, prints a status line
every 5 minutes, and relaunches the driver (at most 3 times) if it stops before the three-seed tables exist. To run
cell 6, stop cell 5 with the stop button (the sweep keeps running), then start cell 5 again. Keep the browser tab
open and the computer awake.

In [ ]:
# 4. Launch or resume the seed sweep in the background
import os, sys, subprocess
os.makedirs("runs/logs", exist_ok=True)
PID_FILE = "runs/logs/seeds_driver.pid"

def driver_alive():
    try:
        pid = int(open(PID_FILE).read().strip())
        os.kill(pid, 0)
        return "Z" not in [l for l in open(f"/proc/{pid}/status") if l.startswith("State:")][0]
    except (OSError, ValueError, IndexError):
        return False              # no pid file, a pid from a previous runtime, or a finished driver

def launch(tag="launch"):
    cmd = [sys.executable, "run.py", "seeds", "--new-seeds", *map(str, NEW_SEEDS), "--recover-interrupted",
           "--workers", str(WORKERS), "--torch-threads", str(TORCH_THREADS), "--rounds", str(ROUNDS),
           "--budgets", *map(str, BUDGETS)]
    log = open("runs/logs/seeds_driver.log", "a")
    log.write(f"\n\n===== {tag} " + " ".join(cmd) + "\n"); log.flush()
    proc = subprocess.Popen(cmd, cwd=WORK, stdout=log, stderr=subprocess.STDOUT, start_new_session=True, env=dict(os.environ))
    open(PID_FILE, "w").write(str(proc.pid))
    return proc.pid

if driver_alive():
    print("A seed-sweep driver is already running in this runtime; see cell 6.")
else:
    print("seed-sweep driver started, pid", launch())

In [ ]:
# 5. Watchdog: keeps this notebook busy and relaunches the driver if it stops before the tables exist.
#    Leave it running. The stop button ends only this loop; the background sweep keeps going.
import os, glob, json, time, datetime
os.chdir(WORK)
MAX_RESTARTS, EVERY_S = 3, 300
TABLES = "runs/study/m14/tables/tables.json"

def sweep_done():
    try:
        return json.load(open(TABLES)).get("seeds") == sorted({0, *NEW_SEEDS})
    except (OSError, ValueError):
        return False

def progress():
    parts = []
    for s in NEW_SEEDS:
        rounds = f"runs/aggregation/m14_s{s}/rounds.json"
        done = len(json.load(open(rounds))) if os.path.exists(rounds) else 0
        loop = "done" if os.path.exists(f"runs/aggregation/m14_s{s}/study.json") else f"{done}/{ROUNDS} rounds"
        ckpts = glob.glob(f"runs/study/m14/direct_ppo_*ee_s{s}/checkpoints/*_steps.zip")
        steps = max((int(p.split("_")[-2]) for p in ckpts), default=0)
        direct = "done" if glob.glob(f"runs/study/m14/direct_ppo_*ee_s{s}/final_model.zip") else f"{100 * steps // (BUDGETS[-1] * 120)} %"
        parts.append(f"s{s}: surrogate {loop}, direct {direct}")
    return "; ".join(parts)

restarts = 0
while not sweep_done():
    now = datetime.datetime.now(datetime.timezone.utc).strftime("%H:%M UTC")
    if not driver_alive():
        if restarts >= MAX_RESTARTS:
            print(f"{now}  driver stopped {restarts} times; not restarting again. Check the logs (cell 6).")
            break
        restarts += 1
        print(f"{now}  driver not running -> relaunch {restarts}/{MAX_RESTARTS}, pid {launch('launch (watchdog)')}")
    else:
        last = open("runs/logs/seeds_driver.log", errors="replace").read().splitlines()[-1:] or [""]
        print(f"{now}  {progress()} | {last[0][:70]}")
    time.sleep(EVERY_S)
else:
    print("three-seed tables written - run cells 7 and 8")

In [ ]:
# 6. Status (stop cell 5 first; restart it afterwards)
import os, glob, json
def tail(path, n=4):
    if os.path.exists(path):
        lines = open(path, errors="replace").read().splitlines()
        print(f"--- {path}"); print("\n".join(lines[-n:]))

print("driver running:", driver_alive())
tail("runs/logs/seeds_driver.log", 6)
for s in NEW_SEEDS:
    for branch in ("surrogate", "direct"):
        tail(f"runs/logs/pipeline_{branch}_s{s}.log", 3)
for s in NEW_SEEDS:
    rounds = f"runs/aggregation/m14_s{s}/rounds.json"
    if os.path.exists(rounds):
        for r in json.load(open(rounds)):
            print(f"  seed {s} round {r['round']}: SUMO V {r['best_sumo_val']:.1f}, cumulative {r['cumulative_ee']} SUMO episodes, "
                  f"PPO {r['ppo_wall_s'] / 60:.0f} min")
    if os.path.exists(f"runs/aggregation/m14_s{s}/study.json"):
        print(f"  seed {s}: aggregation loop finished")
    for run in sorted(glob.glob(f"runs/study/m14/direct_ppo_*ee_s{s}")):
        ckpts = glob.glob(f"{run}/checkpoints/*_steps.zip")
        last = max((int(p.split("_")[-2]) for p in ckpts), default=0)
        total = int(run.split("_")[-2][:-2]) * 120
        print(f"  seed {s} direct PPO: {last} / {total} steps ({100 * last / total:.0f} %)",
              "- finished" if os.path.exists(f"{run}/final_model.zip") else "")
print("final evaluation files:", len(glob.glob("runs/study/m14/eval/*.summary.json")))
print("three-seed tables:", sweep_done())
errors = [p for p in glob.glob("runs/logs/*.log") if "Traceback" in open(p, errors="replace").read()]
print("logs with a Traceback (seed-0 logs may show an old, recovered one):", errors or "none")

In [ ]:
# 7. Tables over seeds 0-2 (per-seed tables: runs/study/m14/tables/seed_<s>/tables.md)
from IPython.display import Markdown, display
path = "runs/study/m14/tables/tables.md"
display(Markdown(open(path).read()) if sweep_done() else Markdown("not built yet (cell 6 shows progress)"))

In [ ]:
# 8. Archive of the results (small files only) next to WORK on Drive
import shutil, datetime, tempfile
keep = ["runs/study/m14/tables", "runs/study/m14/tables_seed0_only", "runs/study/m14/arms.json",
        "runs/study/m14/alinea_tuning.json", "runs/ledger", "runs/logs", "runs/commands.jsonl", "reports", "CODE_VERSION.txt"]
keep += glob.glob("runs/aggregation/m14_s*/rounds.json") + glob.glob("runs/aggregation/m14_s*/study.json")
keep += glob.glob("runs/study/m14/eval/*.jsonl") + glob.glob("runs/study/m14/eval/*.summary.json")
stage = tempfile.mkdtemp()
for item in keep:
    if os.path.isdir(item):
        shutil.copytree(item, os.path.join(stage, item), dirs_exist_ok=True)
    elif os.path.exists(item):
        os.makedirs(os.path.dirname(os.path.join(stage, item)) or stage, exist_ok=True)
        shutil.copy2(item, os.path.join(stage, item))
name = f"{WORK}_seeds_results_{datetime.datetime.now():%Y%m%d_%H%M}"
print(shutil.make_archive(name, "zip", stage))